# Data-Driven Analysis of Text-Conditioned AI-Generated Music: A Case Study with Suno and Udio

In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm 
tqdm.pandas()

import json

import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from matplotlib import pyplot as plt
import webbrowser
sns.set_theme(style="whitegrid")
COLOR_MAP = {'udio': "#e30b5d", 'suno': "#ffbb30"}

import torch
import torch.nn.functional as F

import umap
import umap.umap_ as umap
from collections import Counter
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
spc = spacy.load('en_core_web_sm')

# Load JSON Files
We are not providing our collected data.  
To run this notebook and replicate the methods, data should be collected from Suno and Udio using the links we provide and saved in a folder following the pattern: `./jsons/{suno|udio}/metadata/` 

In [ ]:
from multiprocessing import Pool
from tqdm.auto import tqdm
import os
import pandas as pd
import json

def json2df(json_file):
    df = pd.read_json('./jsons/udio/metadata/' + json_file, orient='index').T
    # print(df['lyrics'])
    return df#.astype(str)#[['id', 'title', 'tags','replaced_tags','prompt',]]

filename = './udio_metadata.pkl'
try:
    udio_df = pd.read_pickle(filename)
    print('opening udio_df from pickle')
    print(udio_df.shape)
except:
    print('creating udio_df from json files')
    udio_metadata_files = os.listdir('./jsons/udio/metadata/')
    print(f'metadata files: {len(udio_metadata_files)}')
    ## buggy for some reasons
    # with Pool(1) as p:
    #     udio_dfs = list(tqdm(p.imap(json2df, udio_metadata_files), total=len(udio_metadata_files)))
    udio_dfs = [pd.read_json('./jsons/udio/metadata/' + json_file, orient='index').T for json_file in tqdm(udio_metadata_files)]
    
    udio_df = pd.concat(udio_dfs).reset_index(drop=True)
    # print(udio_df[['lyrics']].head())
    udio_df = udio_df[['id', 'title','lyrics', 'tags','replaced_tags','prompt']]
    udio_df['source'] = 'udio'
    print(f'saving {filename}')
    # udio_df.to_pickle(filename)

udio_df.head()

In [ ]:
from multiprocessing import Pool
from tqdm.auto import tqdm

def json2df(json_file):
    # return pd.read_json('./jsons/suno/metadata/' + json_file, orient='index').T
    df = pd.json_normalize(json.load(open('./jsons/suno/metadata/' + json_file, 'r')))
    # remove 'metadata,' from column names
    df.columns = [col.replace('metadata.', '') for col in df.columns]
    return df

filename = './suno_metadata.pkl'
try:
    print('opening suno_df from pickle')
    suno_df = pd.read_pickle(filename)
    print(suno_df.shape)
except:
    print('creating suno_df from json files')
    suno_metadata_files = os.listdir('./jsons/suno/metadata/')
    print(f'metadata files: {len(suno_metadata_files)}')
    with Pool(16) as p:
        suno_dfs = list(tqdm(p.imap(json2df, suno_metadata_files), total=len(suno_metadata_files)))
    suno_df = pd.concat(suno_dfs).reset_index(drop=True)
    print(suno_df.columns)
    suno_df = suno_df[['id', 'title', 'tags','gpt_description_prompt','prompt']]
    suno_df['source'] = 'suno'
    print(f'saving {filename}')
    suno_df.to_pickle(filename)

suno_df.head()

# Detect Language in Prompts

- We only consider prompts in English
- We use the fasttext language classifier from huggingface

In [ ]:
print("detecting language...")
import fasttext
from huggingface_hub import hf_hub_download

def detect_language(prompt, model):
    # pandas will convert NaN to string "nan"
    if prompt != "nan" and len(prompt) > 0:
        return model.predict(prompt.replace("\n", " "))[0][0].replace("__label__", "")
    else:
        return None

suno_df['gpt_description_prompt'] = suno_df['gpt_description_prompt'].astype(str)
udio_df['prompt'] = udio_df['prompt'].astype(str)

model_path = hf_hub_download(
    repo_id="facebook/fasttext-language-identification", filename="model.bin"
)
model = fasttext.load_model(model_path)
# model.predict(text)[0]
print("classifying Udio prompts")
udio_df["prompt_language"] = udio_df["prompt"].progress_apply(
    lambda x: detect_language(x, model)
)
print("classifying Suno prompts")
suno_df["prompt_language"] = suno_df["gpt_description_prompt"].progress_apply(
    lambda x: detect_language(x, model)
)
    
print('saving suno metadata to pickle')
suno_df.to_pickle("suno_metadata.pkl")

print('saving udio metadata to pickle')
udio_df.to_pickle("udio_metadata.pkl")

In [ ]:
suno_df['prompt_language'].value_counts()

In [ ]:
udio_df['prompt_language'].value_counts()

# Detect Language in Lyrics

- We only consider lyrics in English
- We use the fasttext language classifier from huggingface

In [ ]:
import pandas as pd
# from tqdm import tqdm
# tqdm.pandas()

REDO = True

def detect_language(prompt, model):
    if prompt != 'nan' and len(prompt) > 0:
        return model.predict(prompt.replace("\n", " "))[0][0].replace("__label__", "")
    else:
        return None

suno_df['prompt'] = suno_df['prompt'].astype(str)
udio_df['lyrics'] = udio_df['lyrics'].astype(str)

# check if dataframes contain language column
if "language" in udio_df.columns and "language" in suno_df.columns and REDO==False:
    print("language column exists")
else:
    print("detecting language...")
    import fasttext
    from huggingface_hub import hf_hub_download

    model_path = hf_hub_download(
        repo_id="facebook/fasttext-language-identification", filename="model.bin"
    )
    model = fasttext.load_model(model_path)
    # model.predict(text)[0]
    print("classifying Udio lyrics")
    udio_df["language"] = udio_df["lyrics"].progress_apply(
        lambda x: detect_language(x, model)
    )
    print("classifying Suno lyrics")
    suno_df["language"] = suno_df["prompt"].progress_apply(
        lambda x: detect_language(x, model)
    )
        
    print('saving suno metadata to pickle')
    suno_df.to_pickle("suno_metadata.pkl")

    print('saving udio metadata to pickle')
    udio_df.to_pickle("udio_metadata.pkl")

In [ ]:
language_df = pd.concat(
    [udio_df[["id", "source", "language"]], suno_df[["id", "source", "language"]]]
).reset_index(drop=True)
# language_df = language_df.groupby(['language']).agg({'source':'value_counts'}).unstack(fill_value=0, level=1).sort_values(by=('source','suno'), ascending=False)

lang_counts = language_df[['language','source']].value_counts().reset_index()
udio_total = lang_counts[lang_counts['source']=='udio']['count'].sum()
suno_total = lang_counts[lang_counts['source']=='suno']['count'].sum()

lang_counts['percentage'] = lang_counts.apply(lambda x: x['count']/udio_total*100 if x['source']=='udio' else x['count']/suno_total*100, axis=1)
print(lang_counts.head(15))

top_langs = lang_counts.groupby('language').sum().sort_values('count', ascending=False)
print("\nTop Languages:\n", top_langs.head(15))
top_langs = top_langs.reset_index()['language']

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format
pd.merge(
    left=lang_counts[(lang_counts['source']=='udio') & (lang_counts['language'].isin(top_langs[:15]))],
    right=lang_counts[(lang_counts['source']=='suno') & (lang_counts['language'].isin(top_langs[:15]))],
    on='language', 
    suffixes=('_udio', '_suno'),
    how='outer'
    ).fillna(0).set_index('language')[['percentage_udio', 'percentage_suno']].loc[top_langs[:15]]


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

g = sns.catplot(
    kind="bar",
    orient="h",
    # data=language_df[language_df["language"].isin(top_langs)], 
    data = lang_counts[lang_counts['language'].isin(top_langs[:15])],
    y="language",
    x="percentage", 
    hue="source",
    # col="source",
    # palette='Set2',
    palette=["#e30b5d", "#ffbb30"],
    hue_order=["udio","suno"],
    saturation=1.0,
    # stat='percent',
    # height=5, 
    aspect=1,
    order=top_langs[:15],
    # width=0.9,
    # legend=True,
    )

plt.show()

In [ ]:
lang_list = lang_counts['language'].unique()
lang_palette = sns.color_palette("Set2", len(lang_list))
lang_colors = dict(zip(lang_list, lang_palette))

sns.set_context('paper')
sns.set_style('whitegrid')

for s in ['suno', 'udio']:
    df = suno_df if s == 'suno' else udio_df
    g = sns.catplot(
        kind='count',
        data=df,
        x="language",
        # stat='percent',
        order=df["language"].value_counts().iloc[:15].index,
        height=4, 
        aspect=3,
        hue='language',
        palette=lang_colors,
        saturation=1.0,
        legend=False,
        ).set_xticklabels(rotation=45).set_axis_labels("Language", "Percentage of Prompts")

    # extract the matplotlib axes_subplot objects from the FacetGrid
    ax = g.facet_axis(0, 0)  # or ax = g.axes.flat[0]
    ax.set_title(f"Language Distribution of Prompts ({s.capitalize()})")

    plt.show()

# Prompts

- Prompts are loaded and filtered by language and length
- Duplicates are dropped to avoid skewing clustering with repeated songs (some users will create multiple version of the same song)
- Textual Embeddings are computed using `NV-Embed-v2`
    - we used the 4bit quantized version 
    - we also compute a selfsimilarity matrix to find close elements if necessary
- We use UMAP to reduce the embeddings to 5 dimensions
- We cluster elements in this space using HDBSCAN
- We reduce the original space down to 2 dimensions for visualization

In [ ]:
def cluster_prompts_bow(column_name, df, top_k=3, verbose=False):
        cluster_names = {}
        for label in tqdm(df[column_name].value_counts().index):
            if verbose: print(f"Cluster {label}")
            if label == '-1':
                cluster_names[label] = 'outliers'
                continue
            else:
                lyrics = []
                for doc in spc.pipe(df[df[column_name] == label]['prompt'], disable=['ner', 'parser', 'textcat', 'word2vec']):
                    lyrics.append(" ".join(token.lemma_ for token in doc))
                
                if verbose: print(len(lyrics))
                
                # do tfidf vectorization
                vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,1))
                X = vectorizer.fit_transform(lyrics)
                word_freq = Counter(dict(zip(vectorizer.get_feature_names_out(), X.sum(axis=0).A1)))
                top3 = word_freq.most_common(top_k)
                if verbose: print(top3)
                cluster_names[label] = '_'.join([t[0] for t in top3])
                
        return cluster_names

In [ ]:
!rm ./prompts_nvemb_hdb.pkl
try:
    # raise Exception("force reload")
    X = pd.read_pickle("./prompts_nvemb_hdb.pkl")
    X = X[X['prompt_language'].str.startswith("eng")]
    X['hdb_label'] = X['hdb_label'].astype(int)
    print(X.shape)
    X = X.drop_duplicates(subset=['prompt'])
    print(X.shape)
    suno_df = pd.read_pickle("suno_metadata.pkl")
    suno_df["source"] = "suno"
    udio_df = pd.read_pickle("udio_metadata.pkl")
    udio_df["source"] = "udio"
    X = X.merge(pd.concat([udio_df, suno_df]), on="id", how="left", suffixes=('', '_y'))
    print(X.shape)
    print(X.columns)
    
except:
    suno_df = pd.read_pickle("suno_metadata.pkl")
    suno_df["source"] = "suno"
    udio_df = pd.read_pickle("udio_metadata.pkl")
    udio_df["source"] = "udio"
    # nltk.download('punkt_tab')
    # nltk.download('stopwords')

    #%% load data
    print("metadata loaded")
    print(f"udio rows: {len(udio_df)}")
    print(f"suno rows: {len(suno_df)}")
    print(f"total rows: {len(udio_df) + len(suno_df)}")

    #%%
    udio_prompts = udio_df[["id", "source", "title","prompt", "language", 'prompt_language']].dropna(subset=["prompt"])
    suno_prompts = suno_df[["id", "source","title","gpt_description_prompt", "language", 'prompt_language'
                            ]].dropna(subset=["gpt_description_prompt"]).rename(columns={"gpt_description_prompt": "prompt"})

    # merge them and assign them a source column
    prompts = pd.concat([udio_prompts, suno_prompts])
    prompts = prompts.dropna(subset=["language"])
    prompts = prompts[prompts['language'].str.startswith("eng") & prompts['prompt_language'].str.startswith("eng") ]
    prompts = prompts[
        (prompts['prompt'].str.len() < 1230) & 
        (prompts['prompt'].str.len() > 0)
        ]
    print(f"total prompts: {len(prompts)}")
    print(prompts.columns)

    X = prompts #.sample(5000)

    #%%
    from transformers import BitsAndBytesConfig, AutoModel
    from torch import float16
    from torch.nn import functional as F
    import os

    os.environ['TOKENIZERS_PARALLELISM'] = "false"

    # quantization_config = BitsAndBytesConfig(
    #     load_in_4bit=True,
    #     bnb_4bit_compute_dtype=float16
    #     )
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_quant_type="nf4", 
        bnb_4bit_use_double_quant=True, 
        bnb_4bit_compute_dtype=torch.bfloat16,
        )

    # load model with tokenizer
    model = AutoModel.from_pretrained(
        'nvidia/NV-Embed-v2', 
        trust_remote_code=True,
        device_map="cuda",
        quantization_config=quantization_config
        # use_8bit=True,
        )

    # df['len_tok'] = df['lyrics'].apply(lambda x: len(model.tokenizer.encode(x)))
    # df = df[df['len_tok'] <= 5000]

    # emb = model.encode(lyrics, instruction="", max_length=max([len(l) for l in lyrics]))
    emb = model._do_encode(
        # model.tokenizer.encode(X['prompt'].values.tolist()).to('cuda', dtype=torch.bfloat16),
        X['prompt'].values.tolist(),
        batch_size=16, 
        instruction="", 
        max_length=500, 
        num_workers=16, 
        return_numpy=False
        )
    emb = F.normalize(emb, p=2, dim=1).detach().to('cpu').numpy()

    X['emb'] = emb.tolist()

    X.to_pickle("prompts_nvemb_embeddings.pkl")

In [ ]:
print("umap 5d")
reducer = umap.UMAP(
        n_jobs=32, 
        n_neighbors=15,
        min_dist=0.15,
        n_components=5,
        metric="cosine",
        # random_state=42, #fixing seed turns off parallelization
    )

X_embedded = reducer.fit_transform(X['emb'].apply(np.array).values.tolist())
print(X_embedded.shape)

for component in range(X_embedded.shape[1]):
    X[f'umap_d{component}'] = X_embedded[:,component]

In [ ]:
#%%
from sklearn.cluster import HDBSCAN

print("clustering")
hdb = HDBSCAN(
    n_jobs=32,
    min_cluster_size=20,
    cluster_selection_method='eom',
    min_samples=20,
    max_cluster_size=250,
    cluster_selection_epsilon=0.25,
)


X['hdb_label'] = hdb.fit_predict(X_embedded)
X['hdb_label'] = X['hdb_label'].astype(str)
# X['hdb_label'].value_counts().reset_index()
# cluster_names = cluster_bow(X, 'hdb_label',top_k=3, verbose=False)
cluster_names = cluster_prompts_bow('hdb_label', X, top_k=3, verbose=False)
X['hdb_names'] = X['hdb_label'].apply(lambda x: cluster_names[x])
X[['hdb_names', 'hdb_label']].value_counts().reset_index()


In [ ]:
import umap.umap_ as umap
print("umap 2d")
reducer = umap.UMAP(
        n_jobs=40,
        n_neighbors=25,
        min_dist=0.65,
        n_components=2,
        metric="cosine",
        # random_state=42,
    )

umap2d = reducer.fit_transform(X['emb'].apply(np.array).values.tolist(), y=X['hdb_label'])
X['umap_x'] = umap2d[:,0]
X['umap_y'] = umap2d[:,1]


In [ ]:
#%%
print("plotting")
cluster_names = {k:v for k,v in X[['hdb_label', 'hdb_names']].values}

scatter = px.scatter(
    X, 
    x="umap_x", 
    y="umap_y", 
    # size = "total", 
    # size = round(np.sqrt(X['total'])),
    color="hdb_names",
    # color_discrete_sequence=px.colors.qualitative.Set2,
    color_discrete_map={"-1":"white", "outliers":"white"},
    # rename labels in legend
    hover_data=["id", "source", "prompt", "hdb_label","title"], 
    category_orders = {"hdb_names": X[['hdb_names', 'hdb_label']].value_counts().reset_index()['hdb_names'].values.tolist()}
    )


fig = go.FigureWidget(layout={'hovermode': 'closest'})
fig.update_layout(width=1600, height=1200)
fig.add_traces(scatter.data)
# fig.update_traces(marker=dict(opacity=0.25))
fig.update_layout(
title="HDBSCAN clusters of prompts embeddings (NV-Embed)",
)

def do_click(trace, points, state):
    if points.point_inds:
        ind = points.point_inds[0]
        # print('ind',ind)
        point_id = trace.customdata[ind][0]
        source = trace.customdata[ind][1]
        print(point_id, source)
        if source == 'suno':
            url = "https://www.suno.com/song/" + point_id
        elif source == 'udio':
            url = "https://www.udio.com/songs/" + point_id
        display(url)
        print(url)
        webbrowser.open_new_tab(url)
[s.on_click(do_click) for s in fig.data]


# for each cluster add centroid with name
# should be on top of the points and not appear in the legend
for label in X['hdb_label'].unique():
    if label == '-1':
        continue
    centroid = X[X['hdb_label'] == label][['umap_x','umap_y']].median()
    fig.add_trace(go.Scattergl(
        x=[centroid['umap_x']],
        y=[centroid['umap_y']],
        mode='text',
        marker=dict(size=12, color='black', symbol='x'),
        text=[cluster_names[label]],
        textposition="top center",
        showlegend=False,
        # zorder=999
    ))



# fig.write_image("./figs/prompts_nvemb_hdb.png")
X.to_pickle("prompts_nvemb_hdb.pkl")
# X[["id","source","prompt","hdb_label","hdb_names","umap_x","umap_y","title"]].to_json('./dash_server/dash_prompts.json', orient='records')

fig#.show()

# Lyrics

- Lyrics are loaded and filtered by language and length
- Duplicates are dropped to avoid skewing clustering with repeated songs (some users will create multiple version of the same song)
- Textual Embeddings are computed using `NV-Embed-v2`
    - we used the 4bit quantized version 
    - we also compute a selfsimilarity matrix to find close elements if necessary
- We use UMAP to reduce the embeddings to 5 dimensions
- We cluster elements in this space using HDBSCAN
- We reduce the original space down to 2 dimensions for visualization

## Load data

In [ ]:
import pandas as pd
import os

REDO = True

if os.path.exists('lyrics_combined.pkl') and not REDO:
    df = pd.read_pickle('lyrics_combined.pkl')
else:
    print('Selecting only songs with english language')
    print('---')

    suno_df = pd.read_pickle("suno_metadata.pkl").dropna(subset=['language'])
    suno_df = suno_df[suno_df['language'].str.startswith('eng_')]
    # suno_df = suno_df.drop_duplicates(subset=['prompt'])
    print(f"Number of English Suno songs: {len(suno_df)}")

    udio_df = pd.read_pickle("udio_metadata.pkl").dropna(subset=['language'])
    udio_df = udio_df[udio_df['language'].str.startswith('eng_')]
    # udio_df = udio_df.drop_duplicates(subset=['lyrics'])
    print(f"Number of English Udio songs: {len(udio_df)}")


    print(f"merging datasets")
    df1 = suno_df[['id','prompt','source','tags','title']].rename(columns={'prompt':'lyrics'})
    df2 = udio_df[['id','lyrics','source','tags','title']]
    df = pd.concat([df1,df2]).reset_index(drop=False)
    print('df shape:',df.shape)
    df['length'] = df['lyrics'].apply(len)
    print('length filtered to [20, 3600]')
    df = df[
        (df['length'] > 20 ) &
        (df['length'] < 3_600)
        ]
    print('dropping duplicates')
    df.drop_duplicates(subset=['lyrics'], inplace=True)
    df.reset_index(drop=True).to_pickle('lyrics_combined.pkl')

df.head()

## Compute embeddings and self-similarity

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from transformers import BitsAndBytesConfig
import numpy as np

if 'emb' in df.columns:
    emb = np.array([np.array(x) for x in df['emb'].values])
else:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
        )

    # load model with tokenizer
    model = AutoModel.from_pretrained(
        'nvidia/NV-Embed-v2', 
        trust_remote_code=True,
        quantization_config=quantization_config
        )

    # df['len_tok'] = df['lyrics'].apply(lambda x: len(model.tokenizer.encode(x)))
    # df = df[df['len_tok'] <= 10_000]

    # emb = model.encode(lyrics, instruction="", max_length=max([len(l) for l in lyrics]))
    emb = model._do_encode(
        # df['lyrics'].values.tolist(),
        ('Title: ' + df['title'] + '\n\n' + df['lyrics']).values.tolist(),
        batch_size=8, 
        instruction="", 
        max_length=3700, # +100 to account for title? also this is probably in tokens not chars 
        num_workers=32, 
        return_numpy=False
        )
    emb = F.normalize(emb, p=2, dim=1).to('cpu').numpy()
    # emb.shape, df.shape
    df['emb'] = emb.tolist()
    # df.head(5)
    df.to_pickle('lyrics_combined.pkl')

if os.path.exists('selfsim.npy'):
    selfsim = np.load('./selfsim.npy')
else:
    selfsim = (emb @ emb.T) * 100
    np.save('./selfsim.npy', selfsim, allow_pickle=True)

### Find similar songs in embeddings space

In [ ]:
import numpy as np

def find_similar(idx, df, selfsim, show_lyrics=False, show_links=False):
    if df.iloc[idx].source == 'suno':
        link = f"https://www.{df.iloc[idx].source}.com/song/{df.loc[idx].id}"
    else:
        link = f"https://www.{df.iloc[idx].source}.com/songs/{df.loc[idx].id}"
        
    print(f"analyzing song {idx} ({link})\n---")

    top5 = np.argsort(selfsim[idx])[::-1][1:6]
    top5 = {i: selfsim[idx][i] for i in top5}
    # top5_idx = list(top5.keys())
    
    for k,v in top5.items():
        if show_links == False:
            print(f"Song:\t{k}\tsim: {v:.2f}")
        elif show_links == True:
            if df.iloc[k].source == 'suno':
                link = f"https://www.{df.iloc[k].source}.com/song/{df.loc[k].id}"
            else:
                link = f"https://www.{df.iloc[k].source}.com/songs/{df.loc[k].id}"
            print(f"Song:\t{k}\tsim: {v:.2f}\tURL:{link}")
    
    if show_lyrics:
        print('showing lyrics for most similar')
        print(f'----- Song {idx} -----')
        print(
            df.loc[idx].lyrics
        )

        most_similar = max(top5, key=top5.get)
        print(f'----- Song {most_similar} -----')
        print(
            df.loc[most_similar].lyrics
        )
        
    return 

In [ ]:
find_similar(
    df[(df['source'] == 'suno') & (df['index'] == 0)].index[0], 
    df, 
    selfsim, 
    show_lyrics=False,
    show_links=True,
    )

## Dimensionality reduction and clustering

In [ ]:
import re
import pandas as pd
import os

if os.path.exists('lyrics_df.pkl'):
    lyrics_df = pd.read_pickle('lyrics_df.pkl')
    cluster_names = {label:name for name,label in lyrics_df[['hdb_names','hdb_label']].values}
else:
    # remove ctrl seqs between square brackets
    df['lyrics'] = df['lyrics'].apply(lambda x: re.sub(r'\[.*?\]', '', x))
    df['lyrics'] = df['lyrics'].apply(str.strip)
    lyrics_df = df[df['lyrics'].str.len() > 1].copy() #.sample()

emb = lyrics_df['emb'].values.tolist()

lyrics_df.head(5)

### UMAP

In [ ]:
# plot using umap
# import umap
import umap.umap_ as umap

reducer = umap.UMAP(
        # n_neighbors=50,
        # min_dist=0.15,
        n_components=5,
        metric="cosine",
        # random_state=42, #fixing seed turns off parallelization
        n_jobs=40, 
    )

for dim, component in enumerate( reducer.fit_transform(emb).T):
    lyrics_df[f'umap_d{dim}'] = component

### HBSCAN Clustering

In [ ]:
from sklearn.cluster import HDBSCAN

hdb = HDBSCAN(
    # min_samples=3,
    # min_cluster_size=20,
    cluster_selection_method='eom',
    # max_cluster_size=200,
    # cluster_selection_epsilon=0.3,
    n_jobs=40,
)

lyrics_df['hdb_label'] = hdb.fit_predict(lyrics_df[[col for col in lyrics_df.columns if col.startswith('umap_d')]])
lyrics_df['hdb_label'] = lyrics_df['hdb_label'].astype(str)
lyrics_df['hdb_label'].value_counts().reset_index()

### Naming clusters

In [ ]:
#import counter
from collections import Counter
from tqdm.auto import tqdm
# import nltk
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
spc = spacy.load('en_core_web_sm')

# nltk.download('punkt_tab')
# nltk.download('stopwords')

def cluster_lyrics_bow(column_name, df, top_k=3, verbose=False):
    cluster_names = {}
    for label in tqdm(df[column_name].value_counts().index):
        if verbose: print(f"Cluster {label}")
        if label == '-1':
            cluster_names[label] = 'outliers'
            continue
        else:
            lyrics = []
            for doc in spc.pipe(df[df[column_name] == label]['lyrics'], disable=['ner', 'parser', 'textcat', 'word2vec']):
                lyrics.append(" ".join(token.lemma_ for token in doc))
            
            if verbose: print(len(lyrics))
            
            # do tfidf vectorization
            vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,1))
            X = vectorizer.fit_transform(lyrics)
            word_freq = Counter(dict(zip(vectorizer.get_feature_names_out(), X.sum(axis=0).A1)))
            top3 = word_freq.most_common(top_k)
            if verbose: print(top3)
            cluster_names[label] = '_'.join([t[0] for t in top3])
            
    return cluster_names

cluster_names = cluster_lyrics_bow('hdb_label',lyrics_df,top_k=3, verbose=False)
lyrics_df['hdb_names'] = lyrics_df['hdb_label'].apply(lambda x: cluster_names[x])
lyrics_df[['hdb_names', 'hdb_label']].value_counts().reset_index()

### Save precomputed dataframe

In [ ]:
lyrics_df.to_pickle('lyrics_df.pkl')

## Plot


In [ ]:
import pandas as pd

lyrics_df = pd.read_pickle('lyrics_df.pkl')
emb = lyrics_df['emb'].values.tolist()

### 2D UMAP

In [ ]:
# import umap
import umap.umap_ as umap

reducer = umap.UMAP(
        n_neighbors=75,
        min_dist=0.25,
        n_components=2,
        metric="cosine",
        # random_state=42,
        n_jobs=40,
    )

X_embedded = reducer.fit_transform(emb, y=lyrics_df['hdb_label'])

lyrics_df['umap_x'] = X_embedded[:,0]
lyrics_df['umap_y'] = X_embedded[:,1]

### Scatterplot with plotly

In [ ]:
import pandas as pd
import numpy as np

import webbrowser
import plotly.express as px
import plotly.graph_objects as go
from IPython import display

# lyrics_df = pd.read_json('../../dash_server/dash_lyrics.json').sample(1000)
print(lyrics_df.shape)

cluster_names = { label: name for name, label in lyrics_df[['hdb_names', 'hdb_label']].values}

scatter = px.scatter(
    lyrics_df,
    x="umap_x",
    y="umap_y",
    color="hdb_names",
    # symbol="source", symbol_map={"suno":"circle","udio":"star-triangle-up"},
    color_discrete_map={'outliers': 'white'},
    hover_data=["id", "source",'title'],
    title="NV-EMBEDv2 lyrics embeddings + HDBSCAN",
    render_mode='webgl',
)
# scatter.show()

fig = go.FigureWidget(layout={'hovermode': 'closest'})
fig.add_traces(scatter.data)
fig.for_each_trace(lambda t: 
    # print(t)
    t.update(marker={
    'opacity' : 0.2 if t['name'] == 'outliers' else 0.4,
    'color' : '#ddd' if t['name'] == 'outliers' else t['marker']['color'],
    'line': {'color': 'grey', 'width': 0.5} if t['name'] == 'outliers' else t['marker']['line'],
    'size': 8
    })
)


def do_click(trace, points, state):
    if points.point_inds:
        ind = points.point_inds[0]
        # print('ind',ind)
        point_id = trace.customdata[ind][0]
        source = trace.customdata[ind][2]
        if source == 'suno':
            url = "https://www.suno.com/song/" + point_id
        else:
            url = "https://www.udio.com/songs/" + point_id
        # display(url)
        print(url)
        webbrowser.open_new_tab(url)
[s.on_click(do_click) for s in fig.data]

## for each cluster add centroid with name
## should be on top of the points and not appear in the legend
for label in lyrics_df['hdb_label'].unique():
    if label == '-1':
        continue
    centroid = lyrics_df[lyrics_df['hdb_label'] == label][['umap_x','umap_y']].mean()
    fig.add_trace(go.Scattergl(
        x=[centroid['umap_x']],
        y=[centroid['umap_y']],
        mode='text',
        marker=dict(size=12, color='black', symbol='x'),
        text= cluster_names[label].split('/')[-1],
        textposition="middle center",
        showlegend=False,
        # zorder=999
    ))
    
    

fig.update_layout(
    # margin=dict(l=0, r=0, t=0, b=0),
    # title=None,
    width=1600, height=800,
    # showlegend=False,
)
# fig.update_xaxes(visible=False, range=[-1.25, 11.25])
# fig.update_yaxes(visible=False, range=[0.25, 13.00])
# fig.update_traces(textfont_size=20)
# # fig.write_image('./figs/lyrics_clean_2x1.pdf')

# fig.update_layout(plot_bgcolor='white')

fig.show()


### 3D version

In [ ]:
reducer = umap.UMAP(
        n_neighbors=50,
        min_dist=0.5,
        n_components=3,
        metric="cosine",
        n_jobs=40,
    )

X_embedded = reducer.fit_transform(emb, y=lyrics_df['hdb_label'])

lyrics_df['umap_3dx'] = X_embedded[:,0]
lyrics_df['umap_3dy'] = X_embedded[:,1]
lyrics_df['umap_3dz'] = X_embedded[:,2]

In [ ]:
# import pandas as pd
# lyrics_df = pd.read_pickle('lyrics_df.pkl')

import webbrowser
import plotly.express as px
import plotly.graph_objects as go
from IPython import display

def do_click(trace, points, state):
    if points.point_inds:
        ind = points.point_inds[0]
        # print('ind',ind)
        point_id = trace.customdata[ind][0]
        # print(point_id)
        if trace.customdata[ind][1].source == 'suno':
            url = "https://www.suno.com/song/" + point_id
        else:
            url = "https://www.udio.com/songs/" + point_id
        # display(url)
        print(url)
        webbrowser.open_new_tab(url)

scatter = px.scatter_3d(
    lyrics_df,
    x="umap_3dx",
    y="umap_3dy",
    z="umap_3dz",
    color="hdb_names",
    # symbol="source", symbol_map={"suno":"circle","udio":"star-triangle-up"},
    color_discrete_map={'outliers': 'white'},
    width=1200, height=800,
    hover_data=["id", "index", "source",'title'],
    title="NV-EMBEDv2 lyrics embeddings + HDBSCAN",
    
).data

fig = go.FigureWidget(layout={'hovermode': 'closest'})
fig.update_layout(title='NV-EMBEDv2 lyrics embeddings + HDBSCAN')
fig.update_layout(width=1600, height=900)
fig.add_traces(scatter)
fig.update_traces(marker=dict(size=2.5, opacity=0.35))
[s.on_click(do_click) for s in fig.data]

# for each cluster add centroid with name
# should be on top of the points and not appear in the legend
for label in lyrics_df['hdb_label'].unique():
    if label == '-1':
        continue
    centroid = lyrics_df[lyrics_df['hdb_label'] == label][['umap_3dx','umap_3dy','umap_3dz']].mean()
    fig.add_trace(go.Scatter3d(
        x=[centroid['umap_3dx']],
        y=[centroid['umap_3dy']],
        z=[centroid['umap_3dz']],
        mode='text',
        marker=dict(size=10, color='black', symbol='x'),
        text=[cluster_names[label]],
        textposition="top center",
        showlegend=False,
        # zorder=999
    ))

fig
# umap.plot.points(reducer, labels=lyrics_df['hdb_names'], width=1000, height=1000, color_key_cmap='tab20')

## Find 5 songs closest to each centroid

In [ ]:
import pandas as pd

lyrics_df = pd.read_pickle('lyrics_df.pkl')

In [ ]:
lyrics_df[['hdb_names','hdb_label']].value_counts().to_csv('cluster_counts.csv')

In [ ]:
import numpy as np

clusters = lyrics_df[['hdb_label','hdb_names']].value_counts().index.to_list()
centroids = lyrics_df.groupby('hdb_names')[['umap_d0','umap_d1','umap_d2','umap_d3','umap_d4']].mean()


with open("top5_clusters.txt","w") as f:
    strings = []
    for label,cluster in clusters:
        if cluster == 'outliers':
            continue
        centroid = centroids.loc[cluster].tolist()
        # print(f"Centroid: {centroid}")
        songs = lyrics_df[lyrics_df['hdb_names'] == cluster][['title','source','id']]
        # distance to centroid
        songs['dist2ctrd'] = songs.apply(lambda x: np.linalg.norm(centroid - lyrics_df.loc[x.name][['umap_d0','umap_d1','umap_d2','umap_d3','umap_d4']]), axis=1)
        top_5 = songs.sort_values('dist2ctrd').reset_index().head(5)
        
        strings.append(f"Cluster: {label} - {cluster}\n")
        
        # iterate top 5 and create urls
        for idx, song in top_5.iterrows():
            if song.source == 'suno': url = f"https://www.suno.com/song/{song.id}"
            elif song.source == 'udio': url = f"https://www.udio.com/songs/{song.id}"
            strings.append(f"{idx+1}) dist: {song.dist2ctrd:.2e} - {url}\n")
        strings.append("\n")
    f.writelines(strings)
        

# Tags

In [ ]:
import pandas as pd 
import re
from tqdm.auto import tqdm

df = pd.read_pickle('lyrics_combined.pkl')[['tags','source']]

In [ ]:

# plot histogram of lengths in 2:1 format
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context('talk')
sns.set_style('whitegrid')
COLOR_MAP = {'udio': "#e30b5d", 'suno': "#ffbb30"}


fig, ax = plt.subplots(1, 1, figsize=(12, 6))

df[df['source'] == 'suno']['tags'].dropna().apply(len).plot.hist(bins=50, alpha=0.5, color=COLOR_MAP['suno'], ax=ax)
df[df['source'] == 'udio']['tags'].dropna().apply(len).plot.hist(bins=50, alpha=0.5, color=COLOR_MAP['udio'], ax=ax)

#show lengend
ax.legend(['Suno', 'Udio'])
# set labels
ax.set_xlabel('Number of tags')
ax.set_ylabel('Number of songs')

plt.show()


## Cleanup

### udio

In [ ]:
udio_tags = []
for t in tqdm(df[df['source']=='udio'].tags.values.sum()):
    # if it does not contain square brackets
        t = re.sub(r'\[|\]', '', t)
        udio_tags.extend(
            [x.lower().strip(".,\n\t+()[] \"") for x in re.split(r'[,.]|(?<=[a-z])/(?=[a-z])', t) if x != '']
        )


    # # if t contains commas split and add to list
    # if ',' not in t:
    #     new_tags.append(t.lower().strip(".,+\n"))
    # else:
    #     for x in t.split(','):
    #         new_tags.append(x.lower().strip(".,\n"))

udio_tags = [t for t in udio_tags if t not in ['',' ']]
print(len(udio_tags))

In [ ]:
vc = pd.DataFrame(udio_tags).value_counts().reset_index().rename(columns={0:'tag'})
print(vc.shape)

print( vc[vc['count'] == 1].shape[0]/vc.shape[0])

# vc[vc['count'] == 3]


In [ ]:
import plotly.express as px

fig = px.bar(
    vc.head(50), 
    x='tag', 
    y='count',
    color_discrete_sequence=['red']#[COLOR_MAP['udio']]
    )
# title
fig.update_layout(title_text='Top 50 Udio Tags')
fig.show()
# fig.write_image('./figs/tags_udio_50.png')

### suno

In [ ]:
suno_tags = []

df[df['source']=='suno']['tags'].dropna().apply(lambda x: suno_tags.extend(re.split(r'[,.]|-\s|(?<=[a-z])/(?=[a-z])', x)))

suno_tags = [x.lower().strip(".,\n\t+()[] \"") for x in suno_tags]
# replace a and an with ''
suno_tags = [re.sub(r'\ba\b|\ban\b', '', x) for x in suno_tags]
suno_tags = [x for x in suno_tags if x not in ['',' ']]

print(len(suno_tags))

In [ ]:
vc = pd.DataFrame(suno_tags).value_counts().reset_index().rename(columns={0:'tag'}).sort_values(by='count', ascending=False)
print(vc.shape)
print(vc[vc['count'] > 1].shape[0]/vc.shape[0])
vc.head(50)
# vc[vc['count'] == 4]

In [ ]:
import plotly.express as px

fig = px.bar(
    vc.head(50), 
    x='tag', 
    y='count',
    color_discrete_sequence=['gold'] #[COLOR_MAP['suno']],
    )
# title
fig.update_layout(title_text='Top 50 Suno Tags')
fig.show()
# fig.write_image('./figs/tags_suno_50.png')

### combine

In [ ]:
#%%
import pandas as pd
from matplotlib import pyplot as plt

tags_counts = pd.merge(
    left = pd.DataFrame(suno_tags, columns=['tags']).assign(source='suno').value_counts(),#.head(250),
    right = pd.DataFrame(udio_tags, columns=['tags']).assign(source='udio').value_counts(),#.head(250),
    on='tags',
    how='outer'
).reset_index().rename(columns={'count_x':'suno','count_y':'udio'}).fillna(0)

tags_counts['total'] = tags_counts['suno'] + tags_counts['udio']

tags_counts['suno_percentage'] = tags_counts['suno'] / tags_counts['suno'].sum() * 100
tags_counts['udio_percentage'] = tags_counts['udio'] / tags_counts['udio'].sum() * 100
tags_counts['percentage_sum'] = tags_counts['suno_percentage'] + tags_counts['udio_percentage']

tags_counts = tags_counts.sort_values('percentage_sum', ascending=False).reset_index(drop=True)
print( tags_counts[tags_counts['total'] == 1].shape[0]/tags_counts.shape[0])
tags_counts

In [ ]:
tags_counts[(tags_counts['tags'].str.contains('\]')) & (tags_counts['total'] > 10)]


In [ ]:
px.bar(
    tags_counts[:50].sort_values('percentage_sum', ascending=False),
    x='tags', 
    y=['udio_percentage','suno_percentage'], 
    # color='source',
    barmode='stack',
    title='Top 50 tags in Suno and Udio songs',
    color_discrete_sequence=['red','gold']#list(COLOR_MAP.values())
    )

In [ ]:
tags_counts.to_pickle('tags_counts.pkl')

## Manual classes

In [ ]:
tags_counts = pd.read_pickle('tags_counts.pkl')
print(tags_counts.shape)
# tags_counts = tags_counts[tags_counts['total'] > 10].dropna(subset='tags').copy() 
tags_counts = tags_counts.head(350).copy()
print(f"Number of tags: {tags_counts.shape[0]}")

In [ ]:

import re
import spacy
spc = spacy.load('en_core_web_sm')

# scrape table from every noise at once
import pandas as pd 

url = 'https://everynoise.com/everynoise1d.html'

enao = pd.read_html(url)[0].rename(columns={0:'index', 1:'icon',2:'genre'})['genre'].values.tolist()
enao += ['electronic dance music','dance-pop','adult contemporary', 'heavy metal', 'northern american music', 'regional music','hip-hop','west coast hip hop','deep bass','heavy bass','neo-psychedelia'] + ['power ballad','progressive electronic','musical theater and entertainment','contemporary christian','doo wop','epic orchestral', 'drum & bass','liquid drum and bass','prog','trip-hop','dance','cappella','dancefloor', 'lo-fi hip hop','neoclassical','alternative','[witch house]','indie','film score','song','808','orchestral','spaghetti western','ballad']
enao = [g.lower() for g in enao]

In [ ]:
def is_genre(tag):
    if tag in enao: return True
    elif tag.__contains__('music'): return True
    else:
        for t in re.split(r'[,./\s-]', tag):
            if t in enao: return True
        return False

tags_counts['is_genre'] = tags_counts['tags'].apply(is_genre)
tags_counts[tags_counts['is_genre']==True]

In [ ]:
def is_adj(tag):
    # use spacy to detect if tag is an adjective or NER
    doc = spc(tag)
    # print([(token.text, token.pos_) for token in doc])
    if len(doc) > 1 or len(doc) == 0:
        return False
    else:
        return doc[0].pos_
    
tags_counts['POS'] = tags_counts['tags'].apply(is_adj)
tags_counts[tags_counts['POS'].notna()]

In [ ]:
instrument_list = [pd.read_html('https://www.imit.org.uk/pages/a-to-z-of-musical-instrument.html')[1][0][2:].values.tolist()]
instrument_list = [x.lower() for x in instrument_list[0]]
instrument_list += ['drums', 'percussion','synth','organ','shamishen','sitar','shakuachi','koto','ukulele']
instrument_list += [x+'s' for x in instrument_list]

def is_instrument(tag):
    if tag in instrument_list:
        return True
    else:
        for t in re.split(r'[,./\s-]', tag):
            if t in instrument_list: return True
        return False
    
tags_counts['is_instrument'] = tags_counts['tags'].apply(is_instrument)

In [ ]:
tags_counts['is_voice'] = tags_counts['tags'].apply(lambda x: bool(sum([str.__contains__(x,y) for y in ['voice', 'vocal','singer']])))
tags_counts['is_bpm'] = tags_counts['tags'].apply(lambda x: bool(re.match(r'\d{2,3}[-\s]?bpm', x)))
tags_counts['is_year'] = tags_counts['tags'].apply(lambda x: bool(re.match(r'\d{2,4}\'?s?', x)))
tags_counts['is_key'] = tags_counts['tags'].apply(lambda x: bool(re.match(r'[a-g]?#?\s?(major|minor)(key)?', x)))

In [ ]:
tags_counts['is_tempo'] = tags_counts['tags'].apply(lambda x: x.__contains__('tempo'))

In [ ]:
struct_keywords = ['verse','chorus','bridge','intro','outro','hook','pre-chorus','post-chorus','interlude','solo','break','drop','refrain','coda','fade','ending','beginning','outro','intro','outro','prelude','melody','melodies']

def is_structure(tag):
    for t in re.split(r'[,./\s-]', tag):
        for kw in struct_keywords:
            if t.__contains__(kw): return True
    return False

tags_counts['is_structure'] = tags_counts['tags'].apply(is_structure)

# tags_counts[tags_counts['is_structure']==True]

In [ ]:
# tags_counts['is_other'] = tags_counts['tags'].apply()

In [ ]:
tags_counts[
    (tags_counts['POS'] == False)
    & (tags_counts['is_genre'] == False) 
    & (tags_counts['is_instrument'] == False)
    & (tags_counts['is_voice'] == False)
    & (tags_counts['is_bpm'] == False) & (tags_counts['is_year'] == False) & (tags_counts['is_key'] == False)
    & (tags_counts['is_structure'] == False)
    & (tags_counts['is_tempo'] == False)
    ]

In [ ]:
def get_labels(row):
    if row['is_genre'] == True: return 'GENRE'
    elif row['is_structure'] == True: return 'STRUCTURE'
    elif row['is_instrument'] == True: return 'INSTRUMENT'
    elif row['is_voice'] == True: return 'VOICE'
    elif row['is_bpm'] == True: return 'BPM'
    elif row['is_year'] == True: return 'YEAR'
    elif row['is_key'] == True: return 'KEY'
    elif row['is_tempo'] == True: return 'TEMPO'
    # if all is none and just pos then 
    elif row['POS'] != None: return 'QUALIFIER'
    # the remainder is other
    else: return 'OTHER'

tags_counts['LABEL'] = tags_counts.apply(get_labels, axis=1)
tags_counts

In [ ]:
tags_counts.to_csv('tags_counts_manual_cat_v2.csv')

In [ ]:
# t1 = pd.read_csv('tags_counts_manual_cat.csv', index_col=0)
# t2 = pd.read_csv('tags_counts_manual_cat_v2.csv', index_col=0)
# xsl = pd.read_excel('tags_counts_manual_cat.xlsx')

# # tags in t2 that are not in t1
# t2[~t2['tags'].isin(t1['tags'])]

# # if row in xsl then t2['LABEL'] = xsl['LABEL']
# t2['LABEL'] = t2.apply(lambda x: xsl[xsl['tags'] == x['tags']]['LABEL'].values[0] if x['tags'] in xsl['tags'].values else x['LABEL'], axis=1)

# t2.to_excel('tags_counts_manual_cat_v2.xlsx')

# print(t2[t2['tags'] == 'introspective'])
# print('---')
# print(xsl[xsl['tags'] == 'introspective'])

## wordclouds

In [ ]:
import pandas as pd
import numpy as np

# tags_counts = pd.read_pickle('./tags_counts.pkl')
# X = pd.read_pickle('tags_counts_top250.pkl')
X = pd.read_excel('tags_counts_manual_cat_v2.xlsx').reset_index(drop=True).sort_values('total', ascending=False)
print(X.shape)
X['tags'] = X.tags.astype(str)

# transform X.label into a unique number
X['label_id'] = X['LABEL'].astype('category').cat.codes
# print(X['label_id'].value_counts())

X.at[175, 'LABEL'] = 'QUALIFIER'
# X[X['tags'] == 'ai']
X.at[907, 'LABEL'] = 'QUALIFIER'

X['LABEL'].value_counts()

In [ ]:
import wordcloud

wc = wordcloud.WordCloud(width=800, height=800, max_words=200, background_color='white')

wc.generate_from_frequencies({x[0]:x[1] for x in X[['tags','suno']].values})
wc.recolor(color_func=wordcloud.get_single_color_func('gold'))
# wc.to_file('./figs/suno_tags_wc.png')
wc.to_image()


In [ ]:
wc.generate_from_frequencies({x[0]:x[1] for x in X[['tags','udio']].values})
wc.recolor(color_func=wordcloud.get_single_color_func('red'))
# wc.to_file('./figs/udio_tags_wc.png')
wc.to_image()

## embed tags with NV-EMBED v2

In [ ]:
import pandas as pd
import numpy as np

# tags_counts = pd.read_pickle('./tags_counts.pkl')
# X = pd.read_pickle('tags_counts_top250.pkl')
X = pd.read_excel('tags_counts_manual_cat_v2.xlsx').reset_index(drop=True).sort_values('total', ascending=False)
print(X.shape)
X['tags'] = X.tags.astype(str)

# transform X.label into a unique number
X['label_id'] = X['LABEL'].astype('category').cat.codes
# print(X['label_id'].value_counts())

X.at[175, 'LABEL'] = 'QUALIFIER'
# X[X['tags'] == 'ai']
X.at[907, 'LABEL'] = 'QUALIFIER'

X['LABEL'].value_counts()

In [ ]:
# !pip install transformers==4.43.4

from transformers import BitsAndBytesConfig, AutoModel
from torch import float16
from torch.nn import functional as F
import os

os.environ['TOKENIZERS_PARALLELISM'] = "false"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=float16
    )

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config=quantization_config,
    )

# df['len_tok'] = df['lyrics'].apply(lambda x: len(model.tokenizer.encode(x)))
# df = df[df['len_tok'] <= 10_000]

# emb = model.encode(lyrics, instruction="", max_length=max([len(l) for l in lyrics]))
emb = model._do_encode(
    X['tags'].values, 
    batch_size=8, 
    instruction="", 
    max_length=500, 
    num_workers=32, 
    return_numpy=False
    )
emb = F.normalize(emb, p=2, dim=1).to('cpu').numpy()
X['nv_emb'] = emb.tolist()

In [ ]:
# X[['total','tags','nv_emb','nv_emb','LABEL']].to_pickle('tags_manual_nvemb.pkl')

### umap tags

In [ ]:
# import numpy as np
# print(np.__version__)
# import pandas as pd

# # X = tags_counts[tags_counts['total'] > 1][['total','tags','emb']].sort_values('total', ascending=False).head(500)
# # print(X.shape)

# # X = pd.read_pickle('tags_counts_nvemb_manual.pkl')
# emb = np.array([np.array(x) for x in X['nv_emb'].values])
# X_embedded = X[['umap_d0','umap_d1','umap_d2','umap_d3','umap_d4']].values


# X[X['tags']  == 'breakup']
# X.at[175, 'LABEL'] = 'QUALIFIER'
# X[X['tags'] == 'ai']
# X.at[907, 'LABEL'] = 'QUALIFIER'

# X.head()

In [ ]:
# plot using umap
import umap
import umap.umap_ as umap
import plotly.express as px

reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.15,
        n_components=5,
        metric="cosine",
        n_jobs=32,
    )

X_embedded = reducer.fit_transform(emb, y=X['label_id'])
print(X_embedded.shape)

for component in range(X_embedded.shape[1]):
    X[f'umap_d{component}'] = X_embedded[:,component]

In [ ]:
from sklearn.cluster import HDBSCAN
from tqdm import tqdm 

# X = pd.read_pickle('tags_counts_nvemb_manual.pkl')

X_embedded = X[['umap_d0','umap_d1','umap_d2','umap_d3','umap_d4']].values

hdb = HDBSCAN(
    # min_samples=5,
    min_cluster_size=5,
    max_cluster_size=50,
    # max_cluster_size=100,
    cluster_selection_epsilon=0.32,
    n_jobs=32,
)

X['hdb_label'] = hdb.fit_predict(X_embedded)
X['hdb_label'] = X['hdb_label'].astype(str)
# X['hdb_label'].value_counts().reset_index()
# cluster_names = cluster_bow(X,'hdb_label',top_k=3, verbose=False)

cluster_names = {}
for label in X['hdb_label'].unique():
    if label == '-1':
        cluster_names[label] = 'OUTLIERS'
        continue
    top3 = X[X['hdb_label'] == label].sort_values('total', ascending=False).head(3)['tags'].values
    words = '_'.join(top3)
    cluster_names[label] = words

X['hdb_names'] = X['hdb_label'].apply(lambda x: cluster_names[x])
X[['hdb_names', 'hdb_label']].value_counts().reset_index()

In [ ]:
import umap.umap_ as umap

emb = X['nv_emb'].values.tolist()

reducer = umap.UMAP(
        n_neighbors=150,
        min_dist=0.45,
        n_components=2,
        metric="cosine",
        n_jobs=32,
    )

umap2d = reducer.fit_transform(emb, y=X['hdb_label'])
X['umap_x'] = umap2d[:,0]
X['umap_y'] = umap2d[:,1]

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# X = pd.read_json('./dash_server/tags_counts_nvemb_manual.json')

fig = px.scatter(
    X, 
    x="umap_x", 
    y="umap_y", 
    # size = "total", 
    size = round(np.power(X['total'], 0.5)),
    color="LABEL",
    opacity=0.5,
    color_discrete_sequence=px.colors.qualitative.Plotly_r,
    # rename labels in legend
    hover_data=["tags", "hdb_label","hdb_names","total"], 
    # title="HDBSCAN clusters of tags embeddings (NV-Embedv2)",
    render_mode='webgl',
    category_orders = {"LABEL": X[['LABEL']].value_counts().reset_index()['LABEL'].values.tolist()}
    )
# fig.update_traces(marker=dict(opacity=0.5))

fig.for_each_trace(
    lambda t: t.update(marker={
        'opacity':[0.25 if x[1] == '-1' else 0.5 for x in t['customdata']],
        'color' : ['#EEE' if x[2] == 'OUTLIERS' else t['marker']['color'] for x in t['customdata']],
        # OUTLIERS show have black outline
        'line' : {'color' : ['grey' if x[2] == 'OUTLIERS' else 'white' for x in t['customdata']],
                   'width' : 0.5
                   },
        'size' : 5 * t['marker']['size']
                   }))

for label in X['hdb_label'].unique():
    if label == '-1':
        continue
    top_tags = X[X['hdb_label'] == label].sort_values('total', ascending=False).head(1)
    centroid = X[X['hdb_label'] == label][['umap_x','umap_y']].mean()
    if label == '40': 
        print('asdas')
        centroid += [-1.0, -5.0]
        
    for t in top_tags.iterrows():
        fig.add_trace(go.Scattergl(
            # x=[t[1]['umap_x']],y=[t[1]['umap_y']],
            x=[centroid['umap_x']], y=[centroid['umap_y']],
            mode='text',
            marker=dict(size=8, color='black'),
            text=t[1]['tags'],
            # font=dict(size=18),
            textposition="middle center",
            showlegend=False,
            # zorder=999
        ))
        
# make text bigger in each trace
fig.update_traces(textfont_size=20)

fig.update_layout(
    width=1800, 
    height=900,
    margin=dict(l=0, r=0, t=0, b=0),
    font=dict(size=18),
    legend=dict(
        # yanchor="top",
        # xanchor="right",
        y=0.01,
        x=0.9,
        title='',
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='rgba(255,255,255,0.5)',
        borderwidth=1,
    )
)
# fig.update_xaxes(visible=False, range = ([-4.5, 7.25]))
# fig.update_yaxes(visible=False, range = ([-30.75, -21.0]))

# change background color to white
fig.update_layout(plot_bgcolor='white')

fig.show()
# fig.write_image('./paper/tags_bgw_2x1.pdf')

# Real artists in prompts/lyrics

## replaced artist names in udio tags

In [ ]:
import pandas as pd
from tqdm import tqdm

udio_df = pd.read_pickle("udio_metadata.pkl")
suno_df = pd.read_pickle("suno_metadata.pkl")

In [ ]:
artist_replaced = []
id_replaced = []

for idx, row in tqdm(udio_df[['replaced_tags']].iterrows()):
    rt = row.values[0]
    if rt == None:
        continue
    for tag in rt.keys():
        if rt[tag]['type'] == 'artist':
            artist_replaced.append(tag)
            id_replaced.append(idx)
            
artist_replaced = pd.DataFrame({'artist': artist_replaced, 'id': id_replaced})
print(artist_replaced.shape)
vc_ar = artist_replaced['artist'].apply(lambda x: x.lower()).value_counts().reset_index()
print(vc_ar.shape)

vc_ar.head(20)


In [ ]:
a,i = artist_replaced.sample(1).values[0]
print(f'artist: {a} - id: {i}')

print('https://udio.com/songs/'+udio_df.loc[i]['id'])
# udio_df.loc[i]


In [ ]:
# suno_sqbr[suno_sqbr['ctrl_seq'].apply(lambda x: x.find('eminem') != -1)]
X = suno_df.dropna(subset='gpt_description_prompt')

# for artist in vc_ar['artist']:
#     prompt_matches = X[X['gpt_description_prompt'].apply(lambda x: x.lower().find(artist) != -1)].shape[0]
#     tags_matches = X[X['tags'].apply(lambda x: x.lower().find(artist) != -1)].shape[0]
#     if prompt_matches > 0 or tags_matches > 0:
#         print(f"{artist} - prompt: {prompt_matches} - tags: {tags_matches}")
    
    
vc_ar['suno_prompt'] = vc_ar['artist'].apply(lambda x: X[X['gpt_description_prompt'].apply(lambda y: y.lower().find(x) != -1)].shape[0])
vc_ar['suno_tags'] = vc_ar['artist'].apply(lambda x: X[X['tags'].apply(lambda y: y.lower().find(x) != -1)].shape[0])

In [ ]:
vc_ar['total'] = vc_ar['suno_prompt'] + vc_ar['suno_tags'] + vc_ar['count']

vc_ar.head(30)

# Meta-tags [Control Sequences]

In [ ]:
import pandas as pd
import re
from tqdm.auto import tqdm
tqdm.pandas()

suno_df = pd.read_pickle("suno_metadata.pkl").dropna(subset=['prompt'])
suno_df['length'] = suno_df['prompt'].apply(len)
suno_df['prompt'] = suno_df['prompt'].apply(lambda x: x.lower())
suno_df = suno_df[(suno_df['length'] > 20 ) & (suno_df['length'] < 3600) & (suno_df['language'] == 'eng_Latn')]
suno_df.head(5)

In [ ]:
# find prompts that contain text between square or round brackets
suno_sqbr = suno_df[suno_df['prompt'].apply(
    # lambda x: len(re.findall(r'\[.*?\]|\(.*?\)', x)) > 0
    lambda x: len(re.findall(r'\[.*?\]', x)) > 0
)]
suno_sqbr['prompt']

In [ ]:
# print(
#     suno_sqbr[suno_sqbr['prompt'].str.find('[chorus:]') != -1]['prompt'].iloc[0]
# )

In [ ]:
ctrl_seqs = []

def find_ctrl_seqs(row, verbose=False):
    # find strings between square or round brackets
    # x = re.findall(r'\[.*?\]|\(.*?\)', row['prompt'])
    x = re.findall(r'\[.*?\]', row['prompt'])
    if verbose: print(x)
    x = [seq.strip("()[]:") for seq in x]
    if len(x) > 0:
        ctrl_seqs.extend(x)
    

_ = suno_sqbr.progress_apply(find_ctrl_seqs, axis=1)

suno_sqbr = pd.Series(ctrl_seqs).apply(str.lower)
suno_sqbr = suno_sqbr.apply(lambda x: re.sub(r'\s+', ' ', x))
suno_sqbr = suno_sqbr.value_counts().reset_index().rename(columns={'index':'ctrl_seq', 0:'count'})

In [ ]:
print(suno_sqbr.shape)
suno_sqbr.head(50)

In [ ]:
suno_sqbr[suno_sqbr['ctrl_seq'].str.contains(r'\W:\W?')]

In [ ]:
udio_df = pd.read_pickle("udio_metadata.pkl").dropna(subset=['lyrics'])
udio_df['length'] = udio_df['lyrics'].apply(len)
udio_df = udio_df[(udio_df['length'] > 20) & (udio_df['length']<3600) &(udio_df['language'] == 'eng_Latn')]

# find prompts that contain square brackets
udio_sqbr = udio_df[udio_df['lyrics'].apply( 
                                            # lambda x: len(re.findall(r'\[.*?\]|\(.*?\)', x)) > 0
                                            lambda x: len(re.findall(r'\[.*?\]', x)) > 0
                                            )]

ctrl_seqs = []

def find_ctrl_seqs(row, verbose=False):
    # find strings between square brackets
    x = re.findall(r'\[.*?\]|\(.*?\)', row['lyrics'])
    x = [seq.strip("()[]:") for seq in x]
    if verbose: print(x)
    if len(x) > 0:
        ctrl_seqs.extend(x)
    

_ = udio_sqbr.progress_apply(find_ctrl_seqs, axis=1)

udio_sqbr = pd.Series(ctrl_seqs).apply(str.lower)
udio_sqbr = udio_sqbr.apply(lambda x: re.sub(r'\s+', ' ', x))
udio_sqbr = udio_sqbr.value_counts().reset_index().rename(columns={'index':'ctrl_seq', 0:'count'})


In [ ]:
print(udio_sqbr.shape)
udio_sqbr.head(50)

In [ ]:
udio_sqbr[udio_sqbr['ctrl_seq'].str.contains(r'\W:\W+')]

In [ ]:
sqbr_df = pd.merge(
    suno_sqbr, udio_sqbr, on='ctrl_seq', how='outer', suffixes=('_suno', '_udio')
).fillna(0)

sqbr_df['total'] = sqbr_df['count_suno'] + sqbr_df['count_udio']
sqbr_df = sqbr_df.sort_values('total', ascending=False)
print(sqbr_df.shape)
# remove numbers from ctrl_seq
sqbr_df['ctrl_seq'] = sqbr_df['ctrl_seq'].apply(lambda x: re.sub(r'\d+', '', x).strip())
print(sqbr_df.shape)
sqbr_df = sqbr_df.groupby('ctrl_seq').sum().sort_values('total', ascending=False)
print(sqbr_df.shape)
# remove empty metatags
sqbr_df = sqbr_df[sqbr_df.index != '']


sqbr_df.head(50)

#### artists in metatags?

In [ ]:
vc_ar['suno_sqbr'] = vc_ar['artist'].apply(lambda x: suno_sqbr[suno_sqbr['ctrl_seq'].apply(lambda y: y.find(x) != -1)].shape[0])
vc_ar['udio_sqbr'] = vc_ar['artist'].apply(lambda x: udio_sqbr[udio_sqbr['ctrl_seq'].apply(lambda y: y.find(x) != -1)].shape[0])

In [ ]:
vc_ar.head(30)

In [ ]:
vc_ar.sort_values('suno_sqbr', ascending=False).head(30)